# 05b · HypCD (Hyp-SimGCD) with a DINOv2 backbone

**HypCD** (Liu, He & Han, CVPR 2025) maps the backbone embedding into a Poincaré ball and trains with both
hyperbolic distance and angle losses; it is the GCD method closest to our question, because hyperbolic space is
meant to encode hierarchy. This notebook trains the official **Hyp-SimGCD** variant on the v2 benchmark.

| | |
|---|---|
| Code | official repo `Visual-AI/HypCD`, pinned commit `ffc895b`, fetched to `files\HypCD\` (MIT licence) |
| Imported from the repo | `ToPoincare`, `HypLinear`, `dist_matrix` (`hyptorch`), `DistillLoss`, `get_params_groups` (`models/model.py`); `SupConLoss` and `info_nce_logits` are read from `train/train_HypSimGCD.py` and executed unchanged |
| Backbone | DINOv2 ViT-B/14 with 4 registers via `torch.hub` (`dinov2_vitb14_reg`), identical to the repo's `dinov2_vitb14_reg4_pretrain.pth` |
| Hyperparameters | the repo's `scripts/train_HypSimGCD.cmd` line for **CUB + DINOv2**: c = 0.1, clip r = 1.2, temperature scale 0.4, max distance weight 1.0, memax 1.0, wd 5e-5, lr 0.1, FP32 |
| Classes and split | identical to Steps 4 and 5a (the same `simgcd_split.json`) |
| Reported | final epoch (no model selection); ACC All/Old/New + per-role NMI/ARI/pair-F1 with the all-singletons baseline |

**Two documented deviations.**
1. The repo uses `xformers` for its own ViT file; installing it would replace this venv's CUDA torch, so the
   identical backbone is loaded through `torch.hub` instead (same architecture, same weights file).
2. The hyperbolic classifier is optimised with `torch.optim.Adam(lr=0.01)` instead of geoopt's `RiemannianAdam`.
   All of that classifier's parameters are ordinary Euclidean tensors, for which RiemannianAdam *is* Adam; this was
   checked numerically against geoopt (max difference 7e-8 after 25 steps). geoopt is avoided because version 0.5.0
   does not import with SciPy ≥ 1.14, and its optimiser state cannot be reloaded by torch ≥ 2.0, which would break
   resuming.

The paper's text quotes c = 0.05 and r = 2.3 for fine-grained data; the released scripts use the values above.
Change them in Step 1 if you prefer the paper's.

**Writes only** to `files\v2\hypcd_dinov2\`. **Order:** as Step 5a.

## Step 1 — Configuration

In [ ]:
# CELL: config
import json, os, sys, time, math, random, inspect, shutil
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

BASE_DIR   = Path(os.environ.get("HNSD_BASE_DIR", r"C:\hnsd\files"))   # data root; set HNSD_BASE_DIR or edit
IMG_ROOT   = BASE_DIR / "selected"              # images: selected/train_mini/<species folder>/*.jpg
V2_DIR     = BASE_DIR / "v2"                    # read only
SPLIT_SRC  = V2_DIR / "simgcd_full" / "simgcd_split.json"   # Step 4 split, reused unchanged
HYPCD_DIR  = BASE_DIR / "HypCD"                 # official HypCD repo (fetched in Step 2)
HYPCD_COMMIT = "ffc895b7ff38be8502bb7e3bd851f1d2a55dff73"
OUT_DIR    = V2_DIR / "hypcd_dinov2"                 # everything this notebook writes

SEED = 42
EPOCHS = 100            # fixed once Step 9 starts (LR, teacher-temperature, distance-weight ramp schedules depend on it)
BATCH_SIZE  = 128
NUM_WORKERS = 6         # set to 0 if Step 8 reports a worker error
EVAL_EVERY  = 10
PRINT_EVERY = 100

DINO_HUB, DINO_MODEL = "facebookresearch/dinov2", "dinov2_vitb14_reg"   # ViT-B/14, 4 registers
DINO_PRETRAINED = True
LR, MOMENTUM, WEIGHT_DECAY = 0.1, 0.9, 5e-5
GRAD_FROM_BLOCK = 11
SUP_WEIGHT      = 0.35
WARMUP_TEACHER_TEMP, TEACHER_TEMP, WARMUP_TEACHER_TEMP_EPOCHS = 0.07, 0.04, 30
N_VIEWS, PROP_TRAIN_LABELS = 2, 0.5
IMAGE_SIZE, CROP_PCT = 224, 0.875
# HypCD, scripts/train_HypSimGCD.cmd, CUB + DINOv2 line: c=0.1, cr=1.2, hts=0.4, hmw=1.0, memax 1.0
MEMAX_WEIGHT     = 1.0
HYP_C            = 0.1     # curvature (paper text: 0.05 for fine-grained)
CLIP_R           = 1.2     # feature clipping radius before expmap0 (paper text: 2.3)
HYPER_TEMP_SCALE = 0.4     # temperature scale of the angle losses
HYPER_MAX_WEIGHT = 1.0     # final weight of the distance losses
HYPER_START_EPOCH, HYPER_END_EPOCH = 0, EPOCHS   # repo: 0 -> epochs (linear ramp over training)
HYP_LR           = 0.01    # RiemannianAdam / Adam lr for the hyperbolic classifier
FP16 = False               # the repo trains in FP32 (no --fp16)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert WARMUP_TEACHER_TEMP_EPOCHS <= EPOCHS
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
print(f"device {DEVICE}" + (f" ({torch.cuda.get_device_name(0)}, "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)" if DEVICE == "cuda" else ""))
print("torch", torch.__version__, "| output folder", OUT_DIR)

## Step 2 — Fetch the official HypCD code and import its components

The repo is downloaded once at the pinned commit. If the download fails, copy `HypCD_ffc895b.zip` (provided with
this notebook) into `files\` and rerun: the cell uses a local zip when present.

In [ ]:
# CELL: method_imports
import ast, zipfile, urllib.request, torchvision, scipy, sklearn

def fetch_hypcd():
    if (HYPCD_DIR / "hyptorch" / "nn.py").exists():
        return "already present"
    zp = BASE_DIR / f"HypCD_{HYPCD_COMMIT[:7]}.zip"
    if not zp.exists():
        for url in [f"https://codeload.github.com/Visual-AI/HypCD/zip/{HYPCD_COMMIT}",
                    f"https://github.com/Visual-AI/HypCD/archive/{HYPCD_COMMIT}.zip"]:
            try:
                urllib.request.urlretrieve(url, zp); break
            except Exception as e:
                print("download failed:", url, e)
    assert zp.exists(), f"put HypCD_{HYPCD_COMMIT[:7]}.zip into {BASE_DIR} and rerun"
    tmp = BASE_DIR / "_hypcd_tmp"
    with zipfile.ZipFile(zp) as z:
        z.extractall(tmp)
    root = next(p for p in tmp.iterdir() if p.is_dir())
    shutil.move(str(root), str(HYPCD_DIR)); shutil.rmtree(tmp, ignore_errors=True)
    return f"extracted {zp.name}"
print("HypCD:", fetch_hypcd(), "->", HYPCD_DIR)

if str(HYPCD_DIR) not in sys.path:
    sys.path.insert(0, str(HYPCD_DIR))
import hyptorch.nn as hypnn
from hyptorch.pmath import dist_matrix
from models.model import DistillLoss, get_params_groups

# SupConLoss and info_nce_logits live inside the training script, which cannot be imported here (it pulls in
# xformers and the dataset code). Their source is read from the file and executed unchanged.
_train_src = (HYPCD_DIR / "train" / "train_HypSimGCD.py").read_text()
_ns = {"torch": torch, "nn": nn, "F": F, "dist_matrix": dist_matrix}
for node in ast.parse(_train_src).body:
    if isinstance(node, (ast.ClassDef, ast.FunctionDef)) and node.name in ("SupConLoss", "info_nce_logits"):
        exec(ast.get_source_segment(_train_src, node), _ns)
SupConLoss, info_nce_logits = _ns["SupConLoss"], _ns["info_nce_logits"]

HYP_OPT = "torch Adam (identical to RiemannianAdam for Euclidean parameters)"
for f in (hypnn.ToPoincare.__init__, hypnn.HypLinear.__init__, SupConLoss.__init__, info_nce_logits, DistillLoss.__init__):
    print(f.__qualname__, inspect.signature(f))
print("hyperbolic classifier optimiser:", HYP_OPT)
print("torchvision", torchvision.__version__, "| scipy", scipy.__version__, "| scikit-learn", sklearn.__version__)

## Step 3 — Class split and labelled subset

The Step 4 split file is copied here unchanged, so SimGCD (DINO), SimGCD (DINOv2) and HypCD all use exactly the
same labelled images.

In [ ]:
# CELL: split
S = json.load(open(V2_DIR / "species_split_v2.json"))
ITEMS = json.load(open(V2_DIR / "items.json"))
ROLE = {s["category_id"]: s["role"] for s in S}
split_path = OUT_DIR / "simgcd_split.json"
if not split_path.exists():
    assert SPLIT_SRC.exists(), f"Step 4 split not found at {SPLIT_SRC}"
    shutil.copyfile(SPLIT_SRC, split_path)
SPLIT = json.load(open(split_path))
old_sp, new_sp = SPLIT["old_species"], SPLIT["new_species"]
assert old_sp == sorted(c for c, r in ROLE.items() if r == "known")
assert new_sp == sorted(c for c, r in ROLE.items() if r in ("novel_known_branch", "novel_unseen_branch"))
CLASS_OF = {c: i for i, c in enumerate(old_sp + new_sp)}
N_OLD, N_CLASSES = len(old_sp), len(old_sp) + len(new_sp)
LAB, UNL = SPLIT["labelled"], SPLIT["unlabelled"]
assert not ({c for _, c in LAB} & set(new_sp)), "LEAKAGE: a new species has labelled images"
missing = [r for r, _ in LAB + UNL if not (IMG_ROOT / r).exists()]
assert not missing, f"{len(missing)} images missing, e.g. {missing[:3]}"
print(f"classes: {N_OLD} old + {N_CLASSES - N_OLD} new = {N_CLASSES}")
print(f"labelled images {len(LAB)} | unlabelled images {len(UNL)} {dict(Counter(ROLE[c] for _, c in UNL))}")

## Step 4 — Datasets in an importable module

Same as Step 4: the dataset lives in a small `.py` file so DataLoader workers work on Windows. The transforms are
SimGCD's (HypCD uses the same `get_transform('imagenet')`).

In [ ]:
# CELL: data_module
DATA_MODULE = r"""
from pathlib import Path
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def get_transforms(image_size=224, crop_pct=0.875):
    # identical to SimGCD / HypCD data/augmentations get_transform('imagenet') with interpolation=3 (bicubic)
    resize = transforms.Resize(int(image_size / crop_pct), InterpolationMode.BICUBIC)
    train = transforms.Compose([resize, transforms.RandomCrop(image_size), transforms.RandomHorizontalFlip(p=0.5),
                                transforms.ColorJitter(), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    test = transforms.Compose([resize, transforms.CenterCrop(image_size), transforms.ToTensor(),
                               transforms.Normalize(MEAN, STD)])
    return train, test

class Views:
    # same as ContrastiveLearningViewGenerator
    def __init__(self, base, n_views=2):
        self.base, self.n_views = base, n_views
    def __call__(self, x):
        return [self.base(x) for _ in range(self.n_views)]

class Items(Dataset):
    # returns (image or list of views, class index, item index, labelled flag)
    def __init__(self, root, rels, labels, transform, labelled_flags=None):
        self.root, self.rels = str(root), list(rels)
        self.labels = np.asarray(labels, dtype=np.int64)
        self.flags = np.zeros(len(self.rels), np.int64) if labelled_flags is None else np.asarray(labelled_flags, np.int64)
        self.transform = transform
    def __len__(self):
        return len(self.rels)
    def __getitem__(self, i):
        with Image.open(Path(self.root) / self.rels[i]) as im:
            img = im.convert("RGB")
        return self.transform(img), int(self.labels[i]), i, int(self.flags[i])
"""
(OUT_DIR / "hnsd_gcd_data.py").write_text(DATA_MODULE.lstrip())
if str(OUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUT_DIR))
import importlib, hnsd_gcd_data as hd
importlib.reload(hd)

train_tf, test_tf = hd.get_transforms(IMAGE_SIZE, CROP_PCT)
TRAIN_SET = hd.Items(IMG_ROOT, [r for r, _ in LAB] + [r for r, _ in UNL],
                     [CLASS_OF[c] for _, c in LAB] + [CLASS_OF[c] for _, c in UNL],
                     hd.Views(train_tf, N_VIEWS), [1] * len(LAB) + [0] * len(UNL))      # labelled first
UNL_TEST = hd.Items(IMG_ROOT, [r for r, _ in UNL], [CLASS_OF[c] for _, c in UNL], test_tf)
UNL_CAT  = np.array([c for _, c in UNL])
UNL_ROLE = np.array([ROLE[c] for c in UNL_CAT])
label_len, unlabelled_len = len(LAB), len(UNL)
SAMPLE_WEIGHTS = torch.DoubleTensor([1.0] * label_len + [label_len / unlabelled_len] * unlabelled_len)

def make_train_loader(num_workers=NUM_WORKERS):
    sampler = torch.utils.data.WeightedRandomSampler(SAMPLE_WEIGHTS, num_samples=len(TRAIN_SET),
                                                     generator=torch.Generator().manual_seed(SEED))
    kw = dict(num_workers=num_workers, pin_memory=(DEVICE == "cuda"))
    if num_workers > 0:
        kw.update(persistent_workers=True, multiprocessing_context="spawn", prefetch_factor=4)
    return torch.utils.data.DataLoader(TRAIN_SET, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True, **kw), sampler

def make_test_loader(ds, num_workers=NUM_WORKERS, batch_size=256):
    kw = dict(num_workers=num_workers)
    if num_workers > 0:
        kw.update(multiprocessing_context="spawn")
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False, **kw)

x, y, i, f = TRAIN_SET[0]
print(f"train set {len(TRAIN_SET)} | views {len(x)} x {tuple(x[0].shape)} | iterations per epoch {len(TRAIN_SET) // BATCH_SIZE}")

## Step 5 — Model

As in `train_HypSimGCD.py`: the DINOv2 backbone (block 11 trainable), `ToPoincare` (clip, then exponential map to
the ball; no parameters) and `HypLinear` (Möbius linear layer, one output per species). The three parts are held
in one module so that checkpoints are simple.

In [ ]:
# CELL: model
def load_backbone():
    bb = torch.hub.load(DINO_HUB, DINO_MODEL, pretrained=DINO_PRETRAINED)
    for p in bb.parameters():
        p.requires_grad = False
    for name, p in bb.named_parameters():              # exactly as train_HypSimGCD.py
        if "block" in name and int(name.split(".")[1]) >= GRAD_FROM_BLOCK:
            p.requires_grad = True
    return bb

class HypModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = load_backbone()
        feat_dim = self.backbone.embed_dim
        # repo: ToPoincare(c, ball_dim=args.mlp_out_dim (256), riemannian=False, clip_r=cr); ball_dim is unused
        self.projector = hypnn.ToPoincare(c=HYP_C, ball_dim=256, riemannian=False, clip_r=CLIP_R)
        self.classifier = hypnn.HypLinear(in_features=feat_dim, out_features=N_CLASSES, c=HYP_C)

def build_model():
    return HypModel().to(DEVICE)

def forward_eval(model, imgs):
    z = model.backbone(imgs); h = model.projector(z); logits = model.classifier(h)
    return z, logits, h.norm(dim=-1)

_m = build_model()
print(f"backbone {DINO_MODEL}: embed {_m.backbone.embed_dim}, registers {getattr(_m.backbone, 'num_register_tokens', 0)}, "
      f"blocks {len(_m.backbone.blocks)} | c {HYP_C} clip_r {CLIP_R}")
print(f"parameters: {sum(p.numel() for p in _m.parameters())/1e6:.1f} M total, "
      f"{sum(p.numel() for p in _m.parameters() if p.requires_grad)/1e6:.1f} M trainable | classes {N_CLASSES}")
del _m
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## Step 6 — Evaluation

ACC All/Old/New with SimGCD/HypCD's `split_cluster_acc_v2` on the unlabelled images; per novel role, NMI, ARI and
pair-F1 with the all-singletons partition beside NMI.

In [ ]:
# CELL: eval
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import normalized_mutual_info_score as NMI, adjusted_rand_score as ARI
from sklearn.metrics.cluster import contingency_matrix

def split_cluster_acc_v2(y_true, y_pred, mask):
    # as in SimGCD / HypCD util/cluster_and_log_utils.py (single-process path)
    y_true = y_true.astype(int)
    old_gt, new_gt = set(y_true[mask]), set(y_true[~mask])
    D = max(y_pred.max(), y_true.max()) + 1
    w = np.zeros((D, D), dtype=np.int64)
    np.add.at(w, (y_pred, y_true), 1)
    ind = np.vstack(linear_sum_assignment(w.max() - w)).T
    ind_map = {j: i for i, j in ind}
    total = sum(w[i, j] for i, j in ind) / y_pred.size
    old = sum(w[ind_map[i], i] for i in old_gt) / max(sum(w[:, i].sum() for i in old_gt), 1)
    new = sum(w[ind_map[i], i] for i in new_gt) / max(sum(w[:, i].sum() for i in new_gt), 1)
    return float(total), float(old), float(new)

def pair_f1(y, p):
    M = contingency_matrix(y, p, sparse=True)
    c2 = lambda x: float((x * (x - 1) / 2).sum())
    tp = c2(M.data.astype(np.float64))
    pp, tt = c2(np.asarray(M.sum(0)).ravel().astype(np.float64)), c2(np.asarray(M.sum(1)).ravel().astype(np.float64))
    pr, rc = (tp / pp if pp else 0.0), (tp / tt if tt else 0.0)
    return (2 * pr * rc / (pr + rc) if pr + rc else 0.0)

@torch.no_grad()
def predict(model, ds, num_workers=NUM_WORKERS, want_feats=False):
    model.eval(); preds, feats, extra = [], [], []
    for imgs, _, _, _ in make_test_loader(ds, num_workers):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(FP16 and DEVICE == "cuda")):
            z, logits, ex = forward_eval(model, imgs)
        preds.append(logits.argmax(1).cpu().numpy())
        if want_feats:
            feats.append(F.normalize(z.float(), dim=-1).cpu().numpy())
            if ex is not None:
                extra.append(ex.float().cpu().numpy())
    cat = lambda L: np.concatenate(L) if L else None
    return np.concatenate(preds), cat(feats), cat(extra)

def evaluate(model, ds=None, cat=None, roles=None, num_workers=NUM_WORKERS):
    ds = UNL_TEST if ds is None else ds
    cat = UNL_CAT if cat is None else cat
    roles = UNL_ROLE if roles is None else roles
    pred, _, _ = predict(model, ds, num_workers)
    y = np.array([CLASS_OF[c] for c in cat])
    out = dict(zip(["ACC_all", "ACC_old", "ACC_new"], split_cluster_acc_v2(y, pred, y < N_OLD)))
    for role in ["novel_known_branch", "novel_unseen_branch"]:
        m = roles == role
        if m.sum() > 1:
            out[role] = {"NMI": NMI(y[m], pred[m]), "ARI": ARI(y[m], pred[m]), "pairF1": pair_f1(y[m], pred[m]),
                         "NMI_singletons": NMI(y[m], np.arange(m.sum())), "n_pred_clusters": int(len(np.unique(pred[m])))}
    return out

def show(res, tag=""):
    s = f"{tag} ACC all {res['ACC_all']:.4f} old {res['ACC_old']:.4f} new {res['ACC_new']:.4f}"
    for role in ["novel_known_branch", "novel_unseen_branch"]:
        if role in res:
            r = res[role]
            s += (f" | {role}: NMI {r['NMI']:.3f} (singletons {r['NMI_singletons']:.3f}) "
                  f"ARI {r['ARI']:.3f} pF1 {r['pairF1']:.3f} clusters {r['n_pred_clusters']}")
    print(s, flush=True)
print("evaluation ready")

## Step 7 — Training loop with resume

The loss is `train_HypSimGCD.py`, line for line: the SimGCD clustering losses on hyperbolic logits, plus
representation losses in two geometries (hyperbolic **distance** with curvature c, and **angle**, i.e. cosine with
a scaled temperature), mixed by a weight that ramps linearly from 0 to `HYPER_MAX_WEIGHT` over training.
Backbone: SGD with cosine LR; hyperbolic classifier: Adam (= RiemannianAdam here), constant lr 0.01.

In [ ]:
# CELL: train_defs
CKPT, LOG = OUT_DIR / "latest.pt", OUT_DIR / "log.json"

def train_step(model, batch, epoch, cluster_criterion):
    images, class_labels, _, mask_lab = batch
    class_labels = class_labels.to(DEVICE, non_blocking=True)
    mask_lab = mask_lab.to(DEVICE, non_blocking=True).bool()
    images = torch.cat(images, dim=0).to(DEVICE, non_blocking=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(FP16 and DEVICE == "cuda")):
        student_out = model.backbone(images)
        student_proj = model.projector(student_out)
        student_out = model.classifier(student_proj)
        teacher_out = student_out.detach()
        # clustering, sup
        sup_logits = torch.cat([f[mask_lab] for f in (student_out / 0.1).chunk(2)], dim=0)
        sup_labels = torch.cat([class_labels[mask_lab] for _ in range(2)], dim=0)
        cls_loss = nn.CrossEntropyLoss()(sup_logits, sup_labels)
        # clustering, unsup
        cluster_loss = cluster_criterion(student_out, teacher_out, epoch)
        avg_probs = (student_out / 0.1).softmax(dim=1).mean(dim=0)
        me_max_loss = - torch.sum(torch.log(avg_probs ** (-avg_probs))) + math.log(float(len(avg_probs)))
        cluster_loss += MEMAX_WEIGHT * me_max_loss
        # represent learning, unsup (hyperbolic distance)
        contrastive_logits, contrastive_labels = info_nce_logits(features=student_proj, hyp_c=HYP_C, normalize=False, device=DEVICE)
        contrastive_loss_distance = torch.nn.CrossEntropyLoss()(contrastive_logits, contrastive_labels)
        # euc unsup loss (angle)
        contrastive_logits_angle, contrastive_labels_angle = info_nce_logits(
            features=student_proj, hyp_c=0, normalize=False, temperature=HYPER_TEMP_SCALE * 1.0, device=DEVICE)
        contrastive_loss_angle = torch.nn.CrossEntropyLoss()(contrastive_logits_angle, contrastive_labels_angle)
        # representation learning, sup
        student_proj = torch.cat([f[mask_lab].unsqueeze(1) for f in student_proj.chunk(2)], dim=1)
        sup_con_labels = class_labels[mask_lab]
        sup_con_loss_distance = SupConLoss(hyp_c=HYP_C)(student_proj, labels=sup_con_labels)
        sup_con_loss_angle = SupConLoss(hyp_c=0, temperature=0.07 * HYPER_TEMP_SCALE)(student_proj, labels=sup_con_labels)
        loss = 0
        loss += (1 - SUP_WEIGHT) * cluster_loss + SUP_WEIGHT * cls_loss
        loss_distance = (1 - SUP_WEIGHT) * contrastive_loss_distance + SUP_WEIGHT * sup_con_loss_distance
        loss_angle = (1 - SUP_WEIGHT) * contrastive_loss_angle + SUP_WEIGHT * sup_con_loss_angle
        lambda_distance = (epoch - (HYPER_START_EPOCH - 1)) / ((HYPER_END_EPOCH - 1) - (HYPER_START_EPOCH - 1))
        lambda_distance = min(1.0, max(0.0, lambda_distance)) * HYPER_MAX_WEIGHT
        loss_rep = (1 - lambda_distance) * loss_angle + lambda_distance * loss_distance
        loss += loss_rep
    return loss, {"cls": cls_loss.item(), "cluster": cluster_loss.item(), "dist_supcon": sup_con_loss_distance.item(),
                  "dist_con": contrastive_loss_distance.item(), "ang_supcon": sup_con_loss_angle.item(),
                  "ang_con": contrastive_loss_angle.item(), "lam": lambda_distance}

def make_optim(model):
    hyp_groups = [{"params": list(model.projector.parameters())}, {"params": list(model.classifier.parameters())}]
    opt_hyp = torch.optim.Adam(hyp_groups, lr=HYP_LR)      # = RiemannianAdam(lr=0.01) for these parameters
    opt = torch.optim.SGD(get_params_groups(model.backbone), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=LR * 1e-3)
    crit = DistillLoss(WARMUP_TEACHER_TEMP_EPOCHS, EPOCHS, N_VIEWS, WARMUP_TEACHER_TEMP, TEACHER_TEMP, hyp_c=0)
    return [opt, opt_hyp], sched, crit

In [ ]:
# CELL: train_loop
def save_ckpt(model, opts, sched, scaler, epoch, hist):
    tmp = CKPT.with_suffix(".tmp")
    torch.save({"epoch": epoch, "model": model.state_dict(), "opts": [o.state_dict() for o in opts],
                "sched": sched.state_dict(), "scaler": scaler.state_dict(), "epochs_total": EPOCHS,
                "torch_rng": torch.get_rng_state(), "np_rng": np.random.get_state()}, tmp)
    os.replace(tmp, CKPT)
    json.dump(hist, open(LOG, "w"), indent=1)

def train_model(max_iters=None, num_workers=NUM_WORKERS, resume=True):
    model = build_model()
    opts, sched, crit = make_optim(model)
    scaler = torch.amp.GradScaler("cuda", enabled=(FP16 and DEVICE == "cuda"))
    start, hist = 0, []
    if resume and CKPT.exists():
        ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
        assert ck["epochs_total"] == EPOCHS, f"checkpoint was made with EPOCHS={ck['epochs_total']}; set EPOCHS back"
        model.load_state_dict(ck["model"])
        for o, s in zip(opts, ck["opts"]):
            o.load_state_dict(s)
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        torch.set_rng_state(ck["torch_rng"].cpu()); np.random.set_state(ck["np_rng"])
        start = ck["epoch"] + 1
        hist = json.load(open(LOG)) if LOG.exists() else []
        print(f"resuming after epoch {start} of {EPOCHS}")
    loader, sampler = make_train_loader(num_workers)
    n_it = len(loader)
    for epoch in range(start, EPOCHS):
        model.train(); sampler.generator.manual_seed(SEED + epoch)
        t0, tot, seen = time.time(), 0.0, 0
        for it, batch in enumerate(loader):
            loss, parts = train_step(model, batch, epoch, crit)
            for o in opts:
                o.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            for o in opts:
                scaler.step(o)
            scaler.update()
            if not math.isfinite(loss.item()):
                raise FloatingPointError(f"loss is {loss.item()} at epoch {epoch+1} iteration {it+1}")
            tot += loss.item(); seen += 1
            if it % PRINT_EVERY == 0 or (max_iters and it + 1 == max_iters):
                rate = (it + 1) * BATCH_SIZE / (time.time() - t0)
                print(f"ep {epoch+1}/{EPOCHS} it {it+1}/{n_it} loss {loss.item():.4f} "
                      + " ".join(f"{k} {v:.3f}" for k, v in parts.items())
                      + f" | {rate:.0f} img/s, epoch ~{n_it*BATCH_SIZE/rate/60:.0f} min", flush=True)
            if max_iters and it + 1 >= max_iters:
                return model, {"img_per_s": seen * BATCH_SIZE / (time.time() - t0), "loss": tot / seen}
        sched.step()
        rec = {"epoch": epoch + 1, "loss": tot / max(seen, 1), "lr": opts[0].param_groups[0]["lr"],
               "minutes": (time.time() - t0) / 60}
        if (epoch + 1) % EVAL_EVERY == 0 or epoch + 1 == EPOCHS:
            rec["eval"] = evaluate(model, num_workers=num_workers); show(rec["eval"], f"[epoch {epoch+1}]")
        hist.append(rec)
        save_ckpt(model, opts, sched, scaler, epoch, hist)
        print(f"epoch {epoch+1} done in {rec['minutes']:.1f} min, mean loss {rec['loss']:.4f}, checkpoint saved", flush=True)
    return model, hist
print("training loop ready")

## Step 8 — Smoke test (about 5 minutes)

30 iterations on a fresh model (nothing saved), then evaluation on 2,000 unlabelled images. Checks workers,
memory and loss, and projects the full run time. **Check the projection before Step 9.**

In [ ]:
# CELL: smoke
SMOKE_ITERS = 30
try:
    _s, info = train_model(max_iters=SMOKE_ITERS, resume=False)
except Exception as e:
    print("ERROR:", type(e).__name__, e)
    print("Worker/pickling error -> NUM_WORKERS = 0 in Step 1. CUDA out of memory -> BATCH_SIZE = 64 (report it).")
    raise
n_it = len(TRAIN_SET) // BATCH_SIZE
ep_min = n_it * BATCH_SIZE / info["img_per_s"] / 60
print(f"\nthroughput {info['img_per_s']:.0f} img/s (2 views each) | one epoch ~{ep_min:.0f} min "
      f"| EPOCHS={EPOCHS}: ~{ep_min*EPOCHS/60:.0f} h of training (+ evaluation every {EVAL_EVERY} epochs)")
for E in (50, 100, 200):
    print(f"   {E:>3} epochs ~ {ep_min*E/60:.0f} h")
if DEVICE == "cuda":
    print(f"peak GPU memory {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
sub = np.random.default_rng(SEED).choice(len(UNL), min(2000, len(UNL)), replace=False)
small = hd.Items(IMG_ROOT, [UNL[i][0] for i in sub], [CLASS_OF[UNL[i][1]] for i in sub], test_tf)
t0 = time.time()
show(evaluate(_s, small, UNL_CAT[sub], UNL_ROLE[sub]), "[smoke eval, 2,000 images]")
print(f"evaluation speed {len(sub)/(time.time()-t0):.0f} img/s")
del _s
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## Step 9 — Full training (resumable)

Safe to interrupt. To continue: rerun Steps 1–7, then this cell. Do not change `EPOCHS` or any hyperparameter
between runs.

In [ ]:
# CELL: train_full
MODEL, HIST = train_model()

## Step 10 — Final evaluation and feature export

Uses the final model. Exports, for every image of the 1,817 benchmark species, the L2-normalised DINOv2 CLS feature after fine-tuning (768-d) and the predicted class index, plus each image's radius in the Poincaré ball (`ball_norm`). With clipping at r, every image whose CLS norm exceeds r lands at the same radius, so `ball_norm` mostly shows whether clipping was active, in the same format as
the other feature files (`X`, `cat_ids`, `relpaths`).

In [ ]:
# CELL: final
if "MODEL" not in dir():
    MODEL = build_model()
    ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
    assert ck["epoch"] + 1 == EPOCHS, f"training is not finished (last epoch {ck['epoch']+1} of {EPOCHS})"
    MODEL.load_state_dict(ck["model"])
FINAL = evaluate(MODEL)
show(FINAL, "[final]")

ALL = [[r, c] for r, c in ITEMS if c in CLASS_OF]
all_ds = hd.Items(IMG_ROOT, [r for r, _ in ALL], [CLASS_OF[c] for _, c in ALL], test_tf)
pred_all, feats, extra = predict(MODEL, all_ds, want_feats=True)
arrays = dict(X=feats.astype(np.float32), cat_ids=np.array([c for _, c in ALL], dtype=np.int64),
              relpaths=np.array([r for r, _ in ALL]), pred_head=pred_all.astype(np.int64))
if extra is not None:
    arrays["ball_norm"] = extra.astype(np.float32)
np.savez(OUT_DIR / f"features_{OUT_DIR.name}_v2.npz", **arrays)
settings = {k: globals()[k] for k in ["EPOCHS", "BATCH_SIZE", "LR", "WEIGHT_DECAY", "SUP_WEIGHT", "MEMAX_WEIGHT",
                                      "FP16", "SEED", "DINO_MODEL", "N_OLD", "N_CLASSES"] if k in globals()}
for k in ["HYP_C", "CLIP_R", "HYPER_TEMP_SCALE", "HYPER_MAX_WEIGHT", "HYP_LR", "HYP_OPT"]:
    if k in globals():
        settings[k] = globals()[k]
json.dump({"final": FINAL, "settings": settings, "labelled": len(LAB), "unlabelled": len(UNL)},
          open(OUT_DIR / "results.json", "w"), indent=1, default=float)
print(f"saved features_{OUT_DIR.name}_v2.npz", feats.shape, "and results.json")

## Step 11 — What to copy to the analysis machine

In [ ]:
# CELL: manifest
want = ["results.json", "log.json", f"features_{OUT_DIR.name}_v2.npz"]
total = 0
for f in want:
    p = OUT_DIR / f
    if p.exists():
        total += p.stat().st_size; print(f"  {f:<36}{p.stat().st_size/1e6:>9.1f} MB")
    else:
        print(f"  {f:<36}  MISSING")
print(f"copy these ({total/1e9:.2f} GB) from {OUT_DIR} to the analysis machine. latest.pt can stay on the GPU machine.")